# ReClip — Colab Whisper API

Bu notebook GPU üzerinde **Whisper large-v3** modelini çalıştırır ve ReClip'in çağırabileceği bir HTTP endpoint açar.

### Adımlar
1. Üst menüden **Runtime > Change runtime type > T4 GPU** seç
2. Hücreleri sırayla çalıştır
3. Son hücredeki `COLAB URL` değerini ReClip Ayarlar > Colab URL alanına yapıştır

### Ngrok (gerekli)
- [ngrok.com](https://ngrok.com) ücretsiz hesap aç → Dashboard > Your Authtoken
- Aşağıdaki `NGROK_TOKEN` değişkenine yapıştır

In [ ]:
# Ngrok token buraya
NGROK_TOKEN = ""  # ngrok.com'dan al (ucretsiz)

# Model secimi: 'large-v3' en iyi kalite | 'large-v3-turbo' biraz daha hizli
WHISPER_MODEL = "large-v3"

In [ ]:
!pip install faster-whisper fastapi uvicorn python-multipart pyngrok -q

In [ ]:
import torch
from faster_whisper import WhisperModel

device = "cuda" if torch.cuda.is_available() else "cpu"
compute = "float16" if device == "cuda" else "int8"
print(f"Cihaz: {device.upper()} | Compute: {compute}")

if device == "cpu":
    print("UYARI: GPU bulunamadi! Runtime > Change runtime type > T4 GPU sec.")

print(f"Model yukleniyor: {WHISPER_MODEL} ...")
model = WhisperModel(WHISPER_MODEL, device=device, compute_type=compute)
print("Model hazir!")

In [ ]:
import os
import tempfile
import threading
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
from pyngrok import ngrok

api = FastAPI(title="ReClip Whisper API")


@api.get("/health")
def health():
    return {"status": "ok", "model": WHISPER_MODEL, "device": device}


@api.post("/transcribe")
async def transcribe(
    file: UploadFile = File(...),
    src_lang: str = Form("auto"),
):
    suffix = os.path.splitext(file.filename or ".mp3")[1] or ".mp3"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(await file.read())
        tmp_path = tmp.name

    try:
        kwargs = {} if src_lang == "auto" else {"language": src_lang}
        segments_gen, info = model.transcribe(
            tmp_path,
            vad_filter=True,
            beam_size=5,
            **kwargs,
        )
        segments = [
            {"start": round(s.start, 3), "end": round(s.end, 3), "text": s.text.strip()}
            for s in segments_gen
            if s.text.strip()
        ]
        return JSONResponse({"segments": segments, "language": info.language})
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)
    finally:
        os.unlink(tmp_path)


# Ngrok tunnel ac
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(8000, bind_tls=True)
public_url = tunnel.public_url

print("\n" + "=" * 55)
print(f"  COLAB URL: {public_url}")
print("=" * 55)
print("Bu URL'yi ReClip > Ayarlar > Colab URL alanina gir.")
print("Notebook acik kaldikca URL gecerli (maks ~12 saat).")
print("=" * 55 + "\n")

# Sunucuyu arka planda baslat
def _run():
    uvicorn.run(api, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=_run, daemon=True).start()
print("Sunucu calisiyor. Bu hucreyi durdurmadin surece aktif kalir.")